# 保底机制 (Guarantee Mechanism) 触发频率调查

调查 `generate_with_dual_cache` 中保底机制在 GSM8K 上的实际触发情况。

**保底机制 (Threshold 策略)**：
- 每步都会强制 unmask 置信度最高的 token（`force_mask`）
- 即使没有 token 通过 threshold，也能保证每步至少 unmask 1 个

**我们要调查的**：
1. `guarantee_added`：保底添加了新 token（max confidence < threshold）的频率
2. `guarantee_only`：保底是**唯一**转移来源的频率（没有任何 token 自然通过 threshold）
3. 触发时的置信度分布
4. 触发在哪些 block / step 中更常见

In [ ]:
import os
import torch
import gc

# ===== 按需修改 =====
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# ====================

os.chdir('llada')

torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
from transformers import AutoTokenizer
from model.modeling_llada import LLaDAModelLM

device = 'cuda'
model_name = 'GSAI-ML/LLaDA-8B-Instruct'

print(f"Loading model: {model_name}")
model = LLaDAModelLM.from_pretrained(
    model_name, trust_remote_code=True, torch_dtype=torch.bfloat16
).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Model loaded!")

## Instrumented 函数

对 `get_transfer_index` 和 `generate_with_dual_cache` 进行 instrumentation，
追踪每一步中保底机制的触发情况。**不修改原版代码**。

In [ ]:
import torch.nn.functional as F
import numpy as np
from generate import add_gumbel_noise, get_num_transfer_tokens


def get_transfer_index_instrumented(logits, temperature, remasking, mask_index, x,
                                     num_transfer_tokens, threshold):
    """
    与原版 get_transfer_index 逻辑完全一致（threshold 分支），
    但额外返回保底机制的触发统计。
    """
    logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
    x0 = torch.argmax(logits_with_noise, dim=-1)

    if remasking == "low_confidence":
        p = F.softmax(logits.to(torch.float64), dim=-1)
        x0_p = torch.gather(p, dim=-1, index=x0.unsqueeze(-1)).squeeze(-1)
    elif remasking == "random":
        x0_p = torch.rand(x0.shape, device=x0.device, dtype=torch.float64)
    else:
        raise NotImplementedError(remasking)

    x0 = torch.where(mask_index, x0, x)
    neg_inf = torch.tensor(torch.finfo(x0_p.dtype).min, device=x0_p.device, dtype=x0_p.dtype)
    confidence = torch.where(mask_index, x0_p, neg_inf)

    # ---- 自然转移（纯 threshold，不含保底） ----
    natural_transfer = mask_index & (confidence >= threshold)
    natural_count = int(natural_transfer.sum().item())

    # ---- 保底：强制 unmask max confidence token ----
    max_conf_indices = torch.argmax(confidence, dim=1, keepdim=True)
    force_mask = torch.zeros_like(natural_transfer).scatter_(1, max_conf_indices, True)
    transfer_index = (natural_transfer | force_mask) & mask_index
    total_count = int(transfer_index.sum().item())

    # ---- 统计 ----
    masked_confs = confidence[mask_index]
    max_conf = float(masked_confs.max().item()) if len(masked_confs) > 0 else 0.0

    stats = {
        'num_masked': int(mask_index.sum().item()),
        'natural_count': natural_count,
        'total_count': total_count,
        # 保底添加了一个不在 natural_transfer 中的 token
        'guarantee_added': total_count > natural_count,
        # 没有任何 token 自然通过 threshold，全靠保底
        'guarantee_only': natural_count == 0,
        'max_confidence': max_conf,
    }

    return x0, transfer_index, stats


@torch.no_grad()
def generate_instrumented(model, prompt, steps=128, gen_length=128, block_length=32,
                          temperature=0., remasking="low_confidence", mask_id=126336,
                          threshold=0.9):
    """
    与原版 generate_with_dual_cache 逻辑一致（threshold 模式），
    但收集每一步的保底触发统计。
    """
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    x = torch.full((B, Lp + gen_length), mask_id, dtype=torch.long, device=model.device)
    x[:, :Lp] = prompt

    nfe = 0
    all_stats = []

    for nb in range(num_blocks):
        s = Lp + nb * block_length
        e = s + block_length

        block_mask_index = (x[:, s:e] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        # Warm-up: full forward
        out_full = model(x, use_cache=True)
        past_key_values = out_full.past_key_values
        nfe += 1

        replace_position = torch.zeros_like(x, dtype=torch.bool)
        replace_position[:, s:e] = True

        # Step 0
        global_mask_index = (x == mask_id)
        global_mask_index[:, e:] = False

        x0, transfer_index, step_stat = get_transfer_index_instrumented(
            out_full.logits, temperature, remasking, global_mask_index, x, None, threshold
        )
        step_stat['block'] = nb
        step_stat['step_in_block'] = 0
        all_stats.append(step_stat)
        x = torch.where(transfer_index, x0, x)

        # Refinement steps
        for i in range(1, steps_per_block):
            if (x[:, s:e] == mask_id).sum() == 0:
                break

            logits_blk = model(
                x[:, s:e], past_key_values=past_key_values,
                use_cache=True, replace_position=replace_position
            ).logits

            mask_blk = (x[:, s:e] == mask_id)

            x0_blk, transfer_idx_blk, step_stat = get_transfer_index_instrumented(
                logits_blk, temperature, remasking, mask_blk, x[:, s:e], None, threshold
            )
            step_stat['block'] = nb
            step_stat['step_in_block'] = i
            all_stats.append(step_stat)

            blk_old = x[:, s:e]
            blk_new = torch.where(transfer_idx_blk, x0_blk, blk_old)
            x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)
            nfe += 1

    return x, nfe, all_stats

print("Instrumented functions defined.")

## 加载 GSM8K 数据 & 构建 5-shot Prompt

In [ ]:
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'  # 禁用 xet-core，避免下载报错

from datasets import load_dataset

ds = load_dataset("gsm8k", "main", split="test")
print(f"GSM8K test set: {len(ds)} samples")

# Standard 5-shot examples (from GSM8K train split, same as lm-eval)
FEW_SHOT = """Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May. Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May. #### 72

Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer: Weng earns 12/60 = <<12/60=0.2>>$0.2 per minute. Working 50 minutes, she earned 0.2 x 50 = <<0.2*50=10>>$10. #### 10

Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to make to buy the wallet?
Answer: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50. Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30. This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more. #### 5

Question: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
Answer: Maila read 12 x 2 = <<12*2=24>>24 pages today. So she was able to read a total of 12 + 24 = <<12+24=36>>36 pages since yesterday. There are 120 - 36 = <<120-36=84>>84 pages left to be read. Since she wants to read half of the remaining pages tomorrow, then she should read 84/2 = <<84/2=42>>42 pages. #### 42

Question: James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?
Answer: He writes each friend 3*2=<<3*2=6>>6 pages a week. So he writes 6*2=<<6*2=12>>12 pages every week. That means he writes 12*52=<<12*52=624>>624 pages a year. #### 624"""


def make_prompt(question):
    """构建 5-shot GSM8K prompt，使用 Instruct 模板。"""
    full = FEW_SHOT.strip() + f"\n\nQuestion: {question}\nAnswer:"
    m = [{"role": "user", "content": full}]
    return tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)


# 验证
sample_prompt = make_prompt(ds[0]['question'])
sample_ids = tokenizer(sample_prompt)['input_ids']
print(f"Sample prompt token length: {len(sample_ids)}")
print(f"Sample question: {ds[0]['question'][:80]}...")

## 运行实验

配置与 `eval_gsm8k.sh` 中 **dual cache + parallel (threshold)** 一致：
- `gen_length=256, block_length=32, steps=256 → steps_per_block=32`
- `threshold=0.9`

`LIMIT` 控制样本数量，按需调整。

In [ ]:
import time

# ===== 实验配置（与 eval_gsm8k.sh dual_cache+threshold 一致） =====
LIMIT = 20            # 样本数量，快速调查用 20，完整用 100+
GEN_LENGTH = 256
BLOCK_LENGTH = 32
STEPS = 256           # dual_cache 模式下 steps=gen_length
THRESHOLD = 0.9
# ================================================================

steps_per_block = STEPS // (GEN_LENGTH // BLOCK_LENGTH)
print(f"Config: limit={LIMIT}, gen_length={GEN_LENGTH}, block_length={BLOCK_LENGTH}")
print(f"        steps={STEPS}, steps_per_block={steps_per_block}, threshold={THRESHOLD}")
print("=" * 60)

all_results = []
start = time.time()

for idx in range(LIMIT):
    question = ds[idx]['question']
    prompt_text = make_prompt(question)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)

    x, nfe, step_stats = generate_instrumented(
        model, input_ids,
        steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
        temperature=0., threshold=THRESHOLD,
    )

    answer = tokenizer.decode(x[0, input_ids.shape[1]:], skip_special_tokens=True)

    # 每个 sample 的聚合
    total_steps = len(step_stats)
    g_added = sum(1 for s in step_stats if s['guarantee_added'])
    g_only  = sum(1 for s in step_stats if s['guarantee_only'])

    all_results.append({
        'idx': idx,
        'question': question[:60],
        'total_steps': total_steps,
        'guarantee_added': g_added,
        'guarantee_only': g_only,
        'nfe': nfe,
        'step_stats': step_stats,
        'answer': answer,
    })

    elapsed = time.time() - start
    eta = elapsed / (idx + 1) * (LIMIT - idx - 1)
    print(f"[{idx+1}/{LIMIT}] steps={total_steps}, "
          f"guarantee_added={g_added} ({g_added/total_steps*100:.0f}%), "
          f"guarantee_only={g_only} ({g_only/total_steps*100:.0f}%), "
          f"nfe={nfe}, ETA={eta:.0f}s")

total_time = time.time() - start
print(f"\nDone! Total time: {total_time:.1f}s ({total_time/LIMIT:.1f}s per sample)")

## 分析结果

In [ ]:
import pandas as pd

# ===== 1. Per-sample summary =====
summary_rows = []
for r in all_results:
    t = r['total_steps']
    summary_rows.append({
        'sample': r['idx'],
        'total_steps': t,
        'guarantee_added': r['guarantee_added'],
        'guarantee_only': r['guarantee_only'],
        'added_rate': f"{r['guarantee_added']/t*100:.1f}%",
        'only_rate': f"{r['guarantee_only']/t*100:.1f}%",
        'nfe': r['nfe'],
    })

df = pd.DataFrame(summary_rows)
print("Per-sample Summary:")
display(df)

# ===== 2. Aggregate =====
total_steps_all = sum(r['total_steps'] for r in all_results)
total_added = sum(r['guarantee_added'] for r in all_results)
total_only = sum(r['guarantee_only'] for r in all_results)

print(f"\n{'='*60}")
print(f"AGGREGATE ({len(all_results)} samples, {total_steps_all} total steps):")
print(f"  guarantee_added (保底添加了新 token):  {total_added} / {total_steps_all}  "
      f"= {total_added/total_steps_all*100:.2f}%")
print(f"  guarantee_only  (保底是唯一来源):      {total_only} / {total_steps_all}  "
      f"= {total_only/total_steps_all*100:.2f}%")
print(f"{'='*60}")

# ===== 3. Per-block analysis =====
block_stats = {}
for r in all_results:
    for s in r['step_stats']:
        b = s['block']
        if b not in block_stats:
            block_stats[b] = {'total': 0, 'added': 0, 'only': 0, 'confs_when_added': []}
        block_stats[b]['total'] += 1
        if s['guarantee_added']:
            block_stats[b]['added'] += 1
            block_stats[b]['confs_when_added'].append(s['max_confidence'])
        if s['guarantee_only']:
            block_stats[b]['only'] += 1

print("\nPer-Block Guarantee Stats:")
print(f"  {'Block':<6} {'Steps':<8} {'Added':<12} {'Only':<12} {'Avg Conf (added)':<18}")
print("-" * 60)
for b in sorted(block_stats.keys()):
    bs = block_stats[b]
    avg_c = np.mean(bs['confs_when_added']) if bs['confs_when_added'] else 0
    print(f"  {b:<6} {bs['total']:<8} {bs['added']:<4} ({bs['added']/bs['total']*100:5.1f}%)  "
          f"{bs['only']:<4} ({bs['only']/bs['total']*100:5.1f}%)  {avg_c:.4f}")

# ===== 4. Per step_in_block analysis =====
sib_stats = {}
for r in all_results:
    for s in r['step_stats']:
        sib = s['step_in_block']
        if sib not in sib_stats:
            sib_stats[sib] = {'total': 0, 'added': 0, 'only': 0}
        sib_stats[sib]['total'] += 1
        if s['guarantee_added']:
            sib_stats[sib]['added'] += 1
        if s['guarantee_only']:
            sib_stats[sib]['only'] += 1

print("\nPer Step-in-Block Guarantee Stats (前 10 步):")
print(f"  {'Step':<6} {'Count':<8} {'Added':<12} {'Only':<12}")
print("-" * 40)
for sib in sorted(sib_stats.keys())[:10]:
    ss = sib_stats[sib]
    print(f"  {sib:<6} {ss['total']:<8} {ss['added']:<4} ({ss['added']/ss['total']*100:5.1f}%)  "
          f"{ss['only']:<4} ({ss['only']/ss['total']*100:5.1f}%)")

## 可视化

In [ ]:
import matplotlib.pyplot as plt

# 汇总所有 step 级别数据
all_step_data = [s for r in all_results for s in r['step_stats']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- (1) 保底触发率 by Block ----
ax = axes[0, 0]
blocks = sorted(block_stats.keys())
added_pct = [block_stats[b]['added'] / block_stats[b]['total'] * 100 for b in blocks]
only_pct  = [block_stats[b]['only']  / block_stats[b]['total'] * 100 for b in blocks]
x_pos = range(len(blocks))
ax.bar(x_pos, added_pct, alpha=0.7, label='guarantee_added', color='orange')
ax.bar(x_pos, only_pct,  alpha=0.7, label='guarantee_only',  color='red')
ax.set_xlabel('Block')
ax.set_ylabel('Trigger Rate (%)')
ax.set_title('Guarantee Trigger Rate by Block')
ax.set_xticks(x_pos)
ax.set_xticklabels(blocks)
ax.legend()

# ---- (2) 置信度分布：触发 vs 未触发 ----
ax = axes[0, 1]
confs_added = [s['max_confidence'] for s in all_step_data if s['guarantee_added']]
confs_normal = [s['max_confidence'] for s in all_step_data if not s['guarantee_added']]
if confs_added:
    ax.hist(confs_added, bins=30, alpha=0.7, color='red',
            label=f'Guarantee triggered (n={len(confs_added)})')
if confs_normal:
    ax.hist(confs_normal, bins=30, alpha=0.7, color='green',
            label=f'Normal (n={len(confs_normal)})')
ax.axvline(x=THRESHOLD, color='black', linestyle='--', label=f'threshold={THRESHOLD}')
ax.set_xlabel('Max Confidence among Masked Tokens')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution')
ax.legend()

# ---- (3) 每步 natural 转移数量分布 ----
ax = axes[1, 0]
natural_counts = [s['natural_count'] for s in all_step_data]
total_counts   = [s['total_count']   for s in all_step_data]
max_c = max(max(total_counts), 10)
bins = range(0, min(max_c + 2, 35))
ax.hist(natural_counts, bins=bins, alpha=0.7, label='Natural (threshold only)', color='steelblue')
ax.hist(total_counts,   bins=bins, alpha=0.4, label='Total (with guarantee)',   color='darkorange')
ax.set_xlabel('Tokens Transferred per Step')
ax.set_ylabel('Count (steps)')
ax.set_title('Tokens Transferred per Step Distribution')
ax.legend()

# ---- (4) 保底触发率 by Step-in-Block ----
ax = axes[1, 1]
sibs = sorted(sib_stats.keys())[:20]  # 前 20 步
sib_rates = [sib_stats[s]['added'] / sib_stats[s]['total'] * 100 for s in sibs]
ax.bar(sibs, sib_rates, alpha=0.7, color='coral')
ax.set_xlabel('Step in Block')
ax.set_ylabel('Guarantee Added Rate (%)')
ax.set_title('Guarantee Trigger Rate by Step Position in Block')

plt.tight_layout()
plt.savefig('guarantee_investigation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to guarantee_investigation.png")

## 详细查看：单个 sample 的逐步追踪

选取一个 guarantee_only 比例较高的样本，查看详细的逐步情况。

In [ ]:
# 找到 guarantee_only 比例最高的 sample
worst = max(all_results, key=lambda r: r['guarantee_only'] / max(r['total_steps'], 1))
print(f"Sample #{worst['idx']}: guarantee_only={worst['guarantee_only']}/{worst['total_steps']} "
      f"({worst['guarantee_only']/worst['total_steps']*100:.1f}%)")
print(f"Question: {worst['question']}...")
print(f"Answer: {worst['answer'][:120]}...")
print()

# 显示该 sample 每一步详情
rows = []
for s in worst['step_stats']:
    rows.append({
        'block': s['block'],
        'step': s['step_in_block'],
        'masked': s['num_masked'],
        'natural': s['natural_count'],
        'total': s['total_count'],
        'g_added': '!' if s['guarantee_added'] else '',
        'g_only': '!!' if s['guarantee_only'] else '',
        'max_conf': f"{s['max_confidence']:.4f}",
    })

df_detail = pd.DataFrame(rows)
print("Step-by-step detail (! = guarantee added new, !! = guarantee only):")
display(df_detail)

## 保存原始数据

保存完整的 step-level 数据到 JSON，方便后续分析。

In [ ]:
import json

# 保存（去掉 step_stats 中的冗余信息，保持文件大小可控）
save_data = {
    'config': {
        'limit': LIMIT, 'gen_length': GEN_LENGTH, 'block_length': BLOCK_LENGTH,
        'steps': STEPS, 'threshold': THRESHOLD,
    },
    'aggregate': {
        'total_steps': total_steps_all,
        'guarantee_added': total_added,
        'guarantee_only': total_only,
        'added_rate': total_added / total_steps_all,
        'only_rate': total_only / total_steps_all,
    },
    'per_sample': [{
        'idx': r['idx'],
        'total_steps': r['total_steps'],
        'guarantee_added': r['guarantee_added'],
        'guarantee_only': r['guarantee_only'],
        'nfe': r['nfe'],
        'step_stats': r['step_stats'],
    } for r in all_results],
}

out_path = f'guarantee_stats_t{THRESHOLD}_n{LIMIT}.json'
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f"Saved to {out_path} ({os.path.getsize(out_path) / 1024:.1f} KB)")